# `generate_simulated_data.ipynb`

Connor Shrader

This file is used to generate model simulations under a variety of different parameter regimes. The resulting data is saved to the `simulated_data/` directory.

### Import packages & define helper functions

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from tqdm import trange, tqdm
from itertools import product
import os

import testis_model as tm
from saving_loading import combine_and_save, check_scenario

In [4]:
def plot_grid(df, scenario):
    fig, ax = plt.subplots(2, 8, figsize = (12, 2.5), dpi = 300)

    ax2 = ax.ravel()
    for i in range(min(n_sims, 16)):
        tm.visualize.plot(data[i], ax = ax2[i])
        ax2[i].set_xlabel('')
        ax2[i].set_ylabel('')
        ax2[i].set_xticks([])
        ax2[i].set_yticks([])

    plt.savefig(f'./scenarios/{scenario}/{scenario}.png')
    plt.close()

    return fig, ax

In [5]:
def sample_ic(n_stem, n_clones, rng = np.random.default_rng()):
    """
    Set initial condition of model.

    This function samples an initial state (s1, s2, ..., sk) where
    k is the number of clones `n_clones` and s1+s2+...+sk = `n_stem`.
    The sampling is done uniformly on this space using stars and bars.
    """
    samples = []

    # Choose `n_clones - 1` cut points. We start at index 1 and use `replace = False`
    # to ensure that each clone has at least one member.
    cuts = rng.choice(range(1, n_stem), size = n_clones - 1, replace = False)
    cuts.sort()

    clone_sizes = [cuts[0]]
    for i in range(1, n_clones - 1):
        clone_sizes.append(cuts[i] - cuts[i - 1])
    clone_sizes.append(n_stem - cuts[-1])

    return np.array(clone_sizes)

In [43]:
sample_times = np.array(list(tm.data_processing.sample_map_int.values()))
print(f'There are {len(sample_times)} sample times.')

There are 18 sample times.


In [51]:
r_event = 36
n_stems = [500, 1000, 1500]
# n_stems = [1000]
r_diffs = [0, 1, 4, 15, 36]
# r_diffs = [1]
# r_events = [0.5, 1, 3, 8]
# p_asymmetrics = [0, 0.5, 1]
r_decays = [3.14, 6.28, 9.42]
n_sims = 100

for n_stem, r_diff, r_decay in product(n_stems, r_diffs, r_decays):
    seed = hash((n_stem, r_diff, r_decay)) % (2**32)
    rng = np.random.default_rng(seed)
    # Model parameters
    # n_stem = 200
    n_clones = rng.binomial(n=18, p=0.5)
    p_asymmetric = 1 - r_diff / 36
    p_diff = q_div = (1 - p_asymmetric) / 2
    sperm_per_diff = 1300
    
    scenario = f'n{n_stem}_r{r_diff}_y{r_decay:.0f}'
    try:
        check_scenario(scenario, overwrite = True)
    except Exception as e: 
        print(f'Skipping {scenario} - file already exists.')
        continue
    
    data = []
    for i in trange(n_sims, desc = f'{scenario}'):
        init_freq = sample_ic(n_stem, n_clones, rng = rng)
        hmm = tm.gillespie.AdvancedTestisModel(
            n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
            q_div = q_div, init_freq = init_freq, r_decay = r_decay, sperm_per_diff = sperm_per_diff
        )

        die_early = rng.random() < 0.5
        n_samples = rng.integers(3, 18) if die_early else 18
        
        sample_times_fish = np.repeat(sample_times[:n_samples], 3)
    
        results, _, _ = hmm.simulate_gillespie(sample_times_fish, rng=rng, n_sperm = 371)
        results.insert(0, 'group', scenario)
        results.insert(1, 'fish', i)
        tm.data_processing.sort_results(results)
        data.append(results)

    df = combine_and_save(data, scenario)
    fig, ax = plot_grid(data, scenario);
    plt.close()

n1500_r36_y9: 100%|███████████████████████████████| 100/100 [01:23<00:00,  1.20it/s]


In [6]:
rng = np.random.default_rng(263894761)

r_event = 36
n_stems = [1000]
r_diffs = [2.5]
# r_events = [0.5, 1, 3, 8]
# p_asymmetrics = [0, 0.5, 1]
r_decays = [6.28]

for n_stem, r_diff, r_decay in product(n_stems, r_diffs, r_decays):
    # Model parameters
    # n_stem = 200
    n_clones = 10
    p_asymmetric = 1 - r_diff / 36
    p_diff = q_div = (1 - p_asymmetric) / 2
    sperm_per_diff = 1300
    
    scenario = f'n{n_stem}_r{r_diff}_y{r_decay:.0f}'
    try:
        check_scenario(scenario, overwrite = True)
    except Exception as e: 
        print(f'Skipping {scenario} - file already exists.')
        continue
    
    data = []
    for i in trange(n_sims, desc = f'{scenario}'):
        init_freq = sample_ic(n_stem, n_clones, rng = rng)
        hmm = tm.gillespie.AdvancedTestisModel(
            n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
            q_div = q_div, init_freq = init_freq, r_decay = r_decay, sperm_per_diff = sperm_per_diff
        )
    
        results, _, _ = hmm.simulate_gillespie(sample_times, rng=rng, n_sperm = 371)
        results.insert(0, 'group', scenario)
        results.insert(1, 'fish', i)
        tm.data_processing.sort_results(results)
        data.append(results)

    df = combine_and_save(data, scenario)
    fig, ax = plot_grid(data, scenario);
    plt.close()

n1000_r2.5_y6: 100%|████████████████████████████| 100/100 [01:32<00:00,  1.08it/s]
